In [2]:
import urllib.request, json, sqlite3, time, csv

API_KEY = "244030413a7348ef96066220165292e2"
BASE = "https://comtradeapi.un.org/data/v1/get/C/A/HS"

# 1. 读取 CSV
countries = {}
with open("european_countries.csv") as f:
    for row in csv.DictReader(f):
        code = row["comtrade_num"].replace(".0", "")
        countries[row["country_name"]] = code

code_to_name = {v: k for k, v in countries.items()}
all_partner_codes = ",".join(countries.values())
print(f"共 {len(countries)} 个国家")

# 2. 建库建表
conn = sqlite3.connect("eu_comtrade.db")
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS trade")
cur.execute("""
CREATE TABLE trade (
    reporter_code  INT,
    reporter_name  TEXT,
    partner_code   INT,
    partner_name   TEXT,
    period         TEXT,
    flow_code      TEXT,
    cmd_code       TEXT,
    primary_value  REAL,
    cif_value      REAL,
    fob_value      REAL,
    net_wgt        REAL,
    partner2_code  INT,
    classification TEXT,
    is_reported    BOOLEAN,
    PRIMARY KEY (reporter_code, partner_code, period, flow_code, cmd_code, partner2_code)
)
""")
cur.execute("CREATE INDEX idx_period ON trade(period)")
cur.execute("CREATE INDEX idx_cmd ON trade(cmd_code)")
cur.execute("CREATE INDEX idx_flow ON trade(flow_code)")
cur.execute("CREATE INDEX idx_partner2 ON trade(partner2_code)")
cur.execute("CREATE INDEX idx_reporter ON trade(reporter_code)")
cur.execute("CREATE INDEX idx_partner ON trade(partner_code)")
conn.commit()

# 3. API 拉取
def fetch_and_store(reporter_code, reporter_name, year):
    params = {
        "reporterCode": reporter_code,
        "period": str(year),
        "partnerCode": all_partner_codes,
        "cmdCode": "AG2",
        "flowCode": "M,X",
        "subscription-key": API_KEY
    }
    query = "&".join(f"{k}={v}" for k, v in params.items())
    url = f"{BASE}?{query}"
    req = urllib.request.Request(url, headers={"Cache-Control": "no-cache"})

    try:
        with urllib.request.urlopen(req) as resp:
            obj = json.loads(resp.read().decode("utf-8"))
    except Exception as e:
        print(f"  ERROR {reporter_name} ({year}): {e}")
        return 0

    count = 0
    for r in obj.get("data", []):
        pname = code_to_name.get(str(r["partnerCode"]), str(r["partnerCode"]))
        cur.execute("""
            INSERT OR REPLACE INTO trade
            VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)
        """, (
            r["reporterCode"], reporter_name,
            r["partnerCode"], pname,
            r["period"], r["flowCode"], r["cmdCode"],
            r["primaryValue"], r.get("cifvalue"), r.get("fobvalue"),
            r.get("netWgt"), r.get("partner2Code", 0),
            r.get("classificationCode"), r.get("isReported")
        ))
        count += 1
    conn.commit()
    return count

# 4. 拉取 2023-2025
years = [2023, 2024, 2025]
total_calls = 0

for year in years:
    for name, code in countries.items():
        n = fetch_and_store(code, name, year)
        total_calls += 1
        print(f"[{total_calls}/150] {name} ({year}): {n} 条")
        time.sleep(1.2)

print(f"\n完成！共 {total_calls} 次调用")
conn.close()

共 50 个国家
[1/150] Norway (2023): 0 条
[2/150] Croatia (2023): 5304 条
[3/150] Italy (2023): 6641 条
[4/150] North Macedonia (2023): 9238 条
[5/150] Serbia (2023): 100000 条
[6/150] Germany (2023): 100000 条
[7/150] Ireland (2023): 5385 条
[8/150] United Kingdom (2023): 100000 条
[9/150] Moldova (2023): 37610 条
[10/150] Greece (2023): 20626 条
[11/150] Turkiye (2023): 100000 条
[12/150] Romania (2023): 100000 条
[13/150] Estonia (2023): 100000 条
[14/150] Switzerland (2023): 0 条
[15/150] Latvia (2023): 34214 条
[16/150] Sweden (2023): 24708 条
[17/150] Faroe Islands (2023): 0 条
[18/150] Slovak Republic (2023): 100000 条
[19/150] Slovenia (2023): 100000 条
[20/150] Gibraltar (2023): 0 条
[21/150] Russian Federation (2023): 0 条
[22/150] Belarus (2023): 0 条
[23/150] Montenegro (2023): 73400 条
[24/150] Liechtenstein (2023): 0 条
[25/150] Iceland (2023): 3897 条
[26/150] Ukraine (2023): 4804 条
[27/150] San Marino (2023): 0 条
[28/150] Greenland (2023): 0 条
[29/150] Netherlands (2023): 7085 条
[30/150] Andorra (20

In [6]:
import pandas as pd

conn = sqlite3.connect("eu_comtrade.db")

df = pd.read_sql("SELECT * FROM trade WHERE reporter_name='Germany' AND period='2024' AND flow_code='X'", conn)
print(df)
print("---------------------------------------------------------------------------")
df2 = pd.read_sql("SELECT * FROM trade WHERE period='2023' AND cmd_code='85'", conn)
print(df2)

df3 = pd.read_sql("""
    SELECT period, flow_code, SUM(primary_value) as total
    FROM trade
    WHERE reporter_name='Italy' AND partner_name='Germany'
    GROUP BY period, flow_code
""", conn)
print(df3)

      reporter_code reporter_name  partner_code partner_name period flow_code  \
0               276       Germany           499   Montenegro   2024         X   
1               276       Germany           380        Italy   2024         X   
2               276       Germany           616       Poland   2024         X   
3               276       Germany            40      Austria   2024         X   
4               276       Germany           642      Romania   2024         X   
...             ...           ...           ...          ...    ...       ...   
6867            276       Germany            40      Austria   2024         X   
6868            276       Germany           380        Italy   2024         X   
6869            276       Germany           233      Estonia   2024         X   
6870            276       Germany           208      Denmark   2024         X   
6871            276       Germany           203      Czechia   2024         X   

     cmd_code  primary_valu

In [7]:
df3 = pd.read_sql("""
    SELECT period, flow_code, SUM(primary_value) as total
    FROM trade
    WHERE reporter_name='Italy' AND partner_name='Germany'
    GROUP BY period, flow_code
""", conn)
print(df3)

  period flow_code         total
0   2023         M  9.697358e+10
1   2023         X  8.071922e+10
2   2024         M  8.718074e+10
3   2024         X  7.390889e+10
4   2025         M  9.669994e+10
5   2025         X  8.158491e+10


In [8]:
# 取双边贸易总额
df = pd.read_sql("""
    SELECT reporter_name, partner_name,
           SUM(primary_value) as total_trade
    FROM trade
    WHERE period = '2023'
    GROUP BY reporter_name, partner_name
""", conn)
print(df)
# 转成矩阵（十亿美元）
matrix = df.pivot_table(
    index='reporter_name',
    columns='partner_name',
    values='total_trade',
    fill_value=0
) / 1e9

# 显示完整表格
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
print(matrix.round(2))

# 贸易额最大的前20对
top = df.nlargest(20, 'total_trade').copy()
top['billion_usd'] = (top['total_trade'] / 1e9).round(2)
print("\n贸易额最大的前20对国家：")
print(top[['reporter_name', 'partner_name', 'billion_usd']].to_string(index=False))

       reporter_name            partner_name   total_trade
0            Albania                 Andorra  1.322154e+04
1            Albania                 Austria  2.099799e+08
2            Albania                 Belarus  2.569231e+06
3            Albania                 Belgium  1.239032e+08
4            Albania  Bosnia and Herzegovina  1.053706e+08
...              ...                     ...           ...
1470  United Kingdom                   Spain  3.224075e+10
1471  United Kingdom                  Sweden  1.127445e+10
1472  United Kingdom                 Turkiye  1.486451e+10
1473  United Kingdom                 Ukraine  1.103968e+09
1474  United Kingdom          United Kingdom  7.635911e+09

[1475 rows x 3 columns]
partner_name            Albania  Andorra  Austria  Belarus  Belgium  \
reporter_name                                                         
Albania                    0.00     0.00     0.21     0.00     0.12   
Andorra                    0.00     0.00     0.01     